<a href="https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane: CTR / Engagement Opportunity Scoring
ML task type: Scoring (ranking-oriented, regression-based) — not classification, not clustering.
Reasoning: The goal of CTR/Engagement Opportunity Scoring is to produce a continuous "opportunity score" per page, so editors get a ranked review queue rather than a binary flag. Classification would force pages into rigid buckets (underperformer / not), losing the nuance of how much opportunity exists. Clustering doesn't apply either, since we're not discovering unlabeled groups — we already know what we're scoring: engagement opportunity relative to peers.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Proxy target: CTR gap = actual_ctr − expected_ctr_for_position_tier
This is the core signal behind CTR/Engagement Opportunity Scoring — expected CTR is the median CTR for a page's position tier (computed in w01). It's a defined proxy, not an observed outcome label, since we don't have real post-fix CTR data. It directly operationalizes "engagement opportunity": how far a page's actual engagement (CTR) falls short of similar-position peers.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: Precision@Top-K — for CTR/Engagement Opportunity Scoring, what matters is whether the top-ranked pages in the opportunity queue are genuinely worth an editor's time, not overall dataset accuracy.
Secondary metric: average CTR-gap magnitude captured in the top decile vs. a random baseline — shows the opportunity-scoring model adds real lift over picking pages randomly.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
!git clone https://github.com/Fizzah-Amir14/flyrank-ml-internship.git repo
%cd repo

Cloning into 'repo'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 139 (delta 50), reused 105 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.85 MiB | 14.43 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/repo


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

vol = df[df["impressions_90d"] >= 100].copy()

# Expected CTR = median CTR of that position tier
tier_median = vol.groupby("position_tier")["ctr"].transform("median")
vol["expected_ctr"] = tier_median
vol["ctr_gap"] = vol["ctr"] - vol["expected_ctr"]  # the CTR/Engagement Opportunity Score

unit_df = vol[["position_tier", "impressions_90d", "ctr", "expected_ctr", "ctr_gap"]]
print("One row = one indexed page, scored for CTR/Engagement Opportunity")
unit_df.sort_values("ctr_gap").head(10)

One row = one indexed page, scored for CTR/Engagement Opportunity


,position_tier,impressions_90d,ctr,expected_ctr,ctr_gap
12651,page_1,155,0.0,0.23,-0.23
10144,page_1,973,0.0,0.23,-0.23
20993,page_1,144,0.0,0.23,-0.23
3751,page_1,683,0.0,0.23,-0.23
10176,page_1,106,0.0,0.23,-0.23
18973,page_1,262,0.0,0.23,-0.23
17490,page_1,678,0.0,0.23,-0.23
4854,page_1,1258,0.0,0.23,-0.23
22356,page_1,370,0.0,0.23,-0.23
22368,page_1,118,0.0,0.23,-0.23


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

CTR/Engagement Opportunity Scoring can't be a fixed rule like "flag under 1% CTR" because CTR varies structurally by position — top_3 tier median CTR is 0.19%, deep tier median is 0.00%. A single threshold would over-flag naturally low-CTR deep pages and miss real underperformers sitting at high positions.
As more signals get folded in later (device, query intent, historical trend), a rule-based system becomes unmanageable. A scoring model can weigh multiple signals together into one smooth, defensible opportunity ranking — which is the whole premise of CTR/Engagement Opportunity Scoring as a lane.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.